In [8]:
%pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached narwhals-2.27.1-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.7.0-py3-none-any.whl.metadata (24 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.2 MB 4.0 MB/s eta 0:00:02
   ------- -------------------------------- 1.6/8.2 MB 4.1 MB/s eta 0:00:02
   ----------- ---------------------------- 2.4/8.2 MB 4.5 MB/s eta 0:00:02
   ---------------- ----------------------- 3.4/8.2 MB 4.3 MB/s eta 0:00:02
   -------------------- ------------------- 4.2/8.2 MB 4.3 MB/s eta 0:00:01
   ------------------------- -------------- 5.2/8.2 MB 4.3 MB/s eta 0:00:01
   ----------------------------- ---------- 6.0/8.2 MB 4.3 MB/s eta 0:00:01
   --------------------------------- ------ 6.8/8.2 MB 4


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
import sklearn

print("Pandas version:", pd.__version__)
print("Scikit-learn version:", sklearn.__version__)

Pandas version: 3.0.6
Scikit-learn version: 1.9.1


In [2]:
import pandas as pd

df = pd.read_csv("../data/processed/customers_clean.csv")

print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

df.head()

Dataset shape: (7043, 21)

Column names:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
print(df["Churn"].value_counts(dropna=False))
print("\nChurn percentages:")
print(
    df["Churn"]
    .value_counts(normalize=True, dropna=False)
    .mul(100)
    .round(2)
)

Churn
No     5174
Yes    1869
Name: count, dtype: int64

Churn percentages:
Churn
No     73.46
Yes    26.54
Name: proportion, dtype: float64


In [4]:
from sklearn.model_selection import train_test_split

# Remove rows where the target is missing
df = df.dropna(subset=["Churn"]).copy()

# Convert churn labels into numbers
y = (
    df["Churn"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        "yes": 1,
        "no": 0,
        "1": 1,
        "0": 0,
        "true": 1,
        "false": 0
    })
)

if y.isna().any():
    raise ValueError("Unexpected values found in the Churn column.")

# Placeholder input: the baseline deliberately ignores features
X = pd.DataFrame({"placeholder": 0}, index=df.index)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("\nTraining churn distribution:")
print(y_train.value_counts(normalize=True).round(3))


Training rows: 5634
Testing rows: 1409

Training churn distribution:
Churn
0    0.735
1    0.265
Name: proportion, dtype: float64


In [5]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

baseline = DummyClassifier(strategy="most_frequent")

baseline.fit(X_train, y_train)

y_pred = baseline.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print(f"Baseline accuracy: {accuracy:.2%}")
print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred, labels=[0, 1]))

Baseline accuracy: 73.46%

Confusion matrix:
[[1035    0]
 [ 374    0]]
